# interface_meta smoke test

Tests the `interface_meta` package: a Python library providing tools for defining interfaces with metadata, version tracking, and runtime validation.

In [ ]:
# Example 1: Define a basic interface via the InterfaceMeta metaclass (2.x API).
from interface_meta import InterfaceMeta

class MyInterface(metaclass=InterfaceMeta):
    INTERFACE_EXPLICIT_OVERRIDES = False

    def greet(self, name):
        """Say hello to name."""
        raise NotImplementedError

class MyImpl(MyInterface):
    def greet(self, name):
        return f"Hello, {name}!"

obj = MyImpl()
result = obj.greet("world")
print(f"greet result: {result}")
assert result == "Hello, world!"
print(f"MyImpl MRO includes MyInterface: {MyInterface in type(obj).__mro__}")

In [ ]:
# Example 2: The override decorator marks explicit overrides.
from interface_meta import InterfaceMeta, override

class Base(metaclass=InterfaceMeta):
    def compute(self, x):
        """Double x."""
        return x * 2

class Child(Base):
    @override
    def compute(self, x):
        return x * 3

c = Child()
print(f"Child.compute(5) = {c.compute(5)}")
assert c.compute(5) == 15
print(f"override callable: {callable(override)}")

In [ ]:
# Example 3: Automatic docstring inheritance (interface_meta 2.x inherits docs
# from the interface method onto conforming overrides).
from interface_meta import InterfaceMeta, override

class DocBase(metaclass=InterfaceMeta):
    def documented_method(self):
        """This is the base docstring."""
        raise NotImplementedError

class DocChild(DocBase):
    @override
    def documented_method(self):
        return "child"

child = DocChild()
print(f"Result: {child.documented_method()}")
print(f"Inherited doc: {DocChild.documented_method.__doc__!r}")
assert child.documented_method() == "child"
assert DocChild.documented_method.__doc__ == "This is the base docstring."

In [ ]:
# Example 4: inherit_docs(method=...) merges an implementation-quirk docstring
# into the interface docs (the 2.x replacement for the old quirk_docs decorator).
from interface_meta import InterfaceMeta, inherit_docs, override

class QuirkBase(metaclass=InterfaceMeta):
    def operate(self):
        """Base operate method."""
        raise NotImplementedError

class QuirkImpl(QuirkBase):
    @override
    @inherit_docs(method='_operate')
    def operate(self):
        return self._operate()

    def _operate(self):
        """Quirk-specific implementation."""
        return "impl-op"

q = QuirkImpl()
print(f"operate() = {q.operate()}")
print(f"merged doc:\n{QuirkImpl.operate.__doc__}")
assert q.operate() == "impl-op"
assert "Quirk-specific implementation" in QuirkImpl.operate.__doc__

In [ ]:
# Example 5: Verify the public API surface exists and has expected names.
import interface_meta

expected_names = ['InterfaceMeta', 'override', 'inherit_docs', 'skip']
available = dir(interface_meta)
found = {n: (n in available) for n in expected_names}

for name, present in found.items():
    print(f"  {name}: {'present' if present else 'MISSING'}")

import importlib.metadata
version = importlib.metadata.version('interface_meta')
print(f"\ninterface_meta version: {version}")

# At minimum, the InterfaceMeta metaclass must be exposed.
assert 'InterfaceMeta' in available, "InterfaceMeta must be exposed"
assert isinstance(interface_meta.InterfaceMeta, type), "InterfaceMeta must be a (meta)class"
assert all(found.values()), f"missing names: {[n for n, p in found.items() if not p]}"
print("\nAll core API names accessible")